# S49_05 — Fine-tuning with Unsloth

**Unsloth** is a library that makes QLoRA fine-tuning 2–3× faster and uses 70% less VRAM than the standard HuggingFace + PEFT stack. It rewrites the forward/backward pass kernels in Triton for common LLM architectures.

## Why Unsloth?

| Method | Speed | VRAM (7B) | Notes |
|--------|-------|-----------|-------|
| Full fine-tune (fp16) | 1× | 80GB+ | Need A100 |
| QLoRA (HuggingFace) | 1× | 12–16GB | Standard |
| QLoRA + Unsloth | 2–3× | 6–9GB | Consumer GPU |
| Unsloth + Flash Attention 2 | 3–5× | 5–7GB | Best |

Unsloth provides drop-in replacements for HuggingFace classes with no code changes beyond the import.

## Complete fine-tuning recipe

In [ ]:
# pip install unsloth[colab] xformers trl peft accelerate bitsandbytes
# (or: pip install unsloth for non-Colab)

unsloth_recipe = '''
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import torch

# --- 1. Load model with Unsloth (drop-in for AutoModelForCausalLM) ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",  # Unsloth hosts pre-quantized models
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)

# --- 2. Add LoRA adapters ---
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,          # Unsloth recommends 0 for speed
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth custom implementation
    random_state=42,
)
model.print_trainable_parameters()

# --- 3. Prepare dataset ---
dataset = load_dataset("json", data_files="training_data.jsonl", split="train")

def format_prompt(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = dataset.map(format_prompt)

# --- 4. Train ---
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="./unsloth-out",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,    # effective batch = 8
        learning_rate=2e-4,
        fp16=False,
        bf16=True,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        dataset_text_field="text",
        max_seq_length=2048,
        logging_steps=10,
    ),
)
trainer.train()

# --- 5. Save adapter and optionally merge ---
model.save_pretrained("./unsloth-adapter")        # LoRA weights only
tokenizer.save_pretrained("./unsloth-adapter")

# Merge and save as 16-bit for serving
# model.save_pretrained_merged("./unsloth-merged", tokenizer, save_method="merged_16bit")

# Save as GGUF for Ollama/llama.cpp deployment
# model.save_pretrained_gguf("./unsloth-gguf", tokenizer, quantization_method="q4_k_m")
'''

print('Unsloth fine-tuning recipe (requires CUDA GPU):')
print(unsloth_recipe)

## Inference after fine-tuning

In [ ]:
inference_code = '''
# Fast inference with Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "./unsloth-adapter",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)  # ~2× faster inference mode

messages = [{"role": "user", "content": "What is gradient descent?"}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    do_sample=True,
)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))
'''
print(inference_code)

## Fine-tuning checklist

Before fine-tuning:
- [ ] Tried prompt engineering and few-shot — still insufficient?
- [ ] Have ≥500 high-quality instruction pairs?
- [ ] Have a held-out evaluation set (10–20% of data)?
- [ ] Defined a clear metric for success?

During training:
- [ ] Monitor training loss (should decrease smoothly)
- [ ] Monitor eval loss (should also decrease; if it rises = overfitting)
- [ ] Check sample outputs after training — does format look right?

After training:
- [ ] Evaluate on your held-out set with your metric
- [ ] Compare to prompted baseline (sometimes prompting wins)
- [ ] Test for regressions — capabilities the base model had that fine-tuned model lost

> **2026 context:** Unsloth supports Llama 3, Mistral, Phi-4, Gemma, and Qwen with optimized kernels. The Unsloth Pro version adds 4–10× speedups. For most practitioners, `unsloth/` model IDs on HuggingFace Hub are the easiest starting point.

This completes S49. Next section: [S50_LLMOps](../S50_LLMOps/S50_01_quantization.ipynb)